# A549 Supplementary LOY/TreeMet Figures

Reviewer-ready analysis of LOY/ROY and TreeMet metrics using the integrated dataset.


## 1. Environment and versions

In [ ]:
# Environment + determinism (set before importing heavy libs)
import os

# Best-effort determinism across machines (threads can introduce tiny non-determinism)
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')

import scanpy as sc
import harmonypy
import leidenalg
import igraph
import umap

from importlib.metadata import PackageNotFoundError, version


def _pkg_version(dist_name: str, module=None) -> str:
    try:
        return version(dist_name)
    except PackageNotFoundError:
        return getattr(module, "__version__", "unknown")


sc.settings.n_jobs = 1
sc.logging.print_header()
print(f"scanpy: {_pkg_version('scanpy', sc)}")
print(f"harmonypy: {_pkg_version('harmonypy', harmonypy)}")
print(f"leidenalg: {_pkg_version('leidenalg', leidenalg)}")
print(f"python-igraph: {_pkg_version('igraph', igraph)}")
print(f"umap-learn: {_pkg_version('umap-learn', umap)}")


This notebook contains **supplementary** plots (QC/UMAP/score visualizations and correlation plots).
Main Figure panels **D–F** are generated in `02_A549_LOY_TreeMet_Analysis.ipynb`.

## 2. Parameters

In [ ]:
# Scoring
MIN_GENES_FOR_SCORING = 3
CTRL_SIZE = 50

# LOY/ROY
LOY_THRESHOLD = -0.2
GROUP_KEY = 'Sample2'
REFERENCE_GROUP = 'LL'

# Correlation plots
CORR_METHOD = 'spearman'
X_LIMITS = (-0.5, 0.7)
Y_LIMITS = (-0.1, 0.6)
EXCLUDE_GROUPS = ['LM0_Engraft', 'LM0_NotEngraft']

# Reproducibility
RANDOM_SEED = 0
import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 3. Paths

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
for p in [CWD, *CWD.parents]:
    if (p / "code" / "figure6" / "scripts" / "project_paths.py").exists():
        REPO_ROOT = p
        break
else:
    raise RuntimeError(
        "Could not locate the repository root. Launch Jupyter from within the repo (or a subdirectory) so code/figure6/notebooks is visible."
    )

if str(REPO_ROOT / "code" / "figure6") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "code" / "figure6"))

from scripts.project_paths import get_notebook_paths

PATHS = get_notebook_paths("03_FigS6_Supplementary", REPO_ROOT)
PROJECT_ROOT = PATHS.common.repo_root
DIR_CODE = PATHS.common.code_dir
DIR_DATA = PATHS.common.data_dir
DIR_RUNTIME = PATHS.common.runtime_dir
DIR_FIGURES_ROOT = PATHS.common.figures_root
DIR_SAVE = PATHS.save_dir
DIR_FIG_SAVE = PATHS.figure_dir
DIR_PROC_IN = PATHS.common.processed_dir

sc.settings.verbosity = 3
sc.settings.figdir = str(DIR_FIG_SAVE)


## 4. Imports and helper functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.utils import (
    add_loy_roy_labels,
    add_loy_roy_proportions,
    build_s6h_clone_enrichment_table,
    build_loy_roy_table,
    chi_square_loy_roy,
    fisher_pairwise_vs_reference,
    plot_score_correlation,
    summarize_s6h_clone_phenotypes,
)


def my_savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"[Saved] {path}")
    if path.suffix.lower() == '.pdf':
        png_path = path.with_suffix('.png')
        plt.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"[Saved] {png_path}")


## 5. Load integrated dataset

In [ ]:
adata = sc.read(DIR_PROC_IN / 'A549_integrated_final.h5ad')
print(adata)


## 6. Gene signature scoring
Score Y, X, and housekeeping gene sets.

In [ ]:
GENE_SIGNATURES = {
    'Y_score': [
        'EIF1AY','KDM5D','TXLNGY','RPS4Y1','UTY','DDX3Y','ZFY','PRKY','TBL1Y',
        'NLGN4Y','USP9Y','TMSB4Y','PCDH11Y'
    ],
    'X_score': [
        'EIF1AX','KDM5C','TXLNG','RPS4X','KDM6A','DDX3X','ZFX','PRKX','TBL1X',
        'NLGN4X','USP9X','TMSB4X','PCDH11X'
    ],
    'Housekeep_score': [
        'RPLP0', 'RPL13A', 'RPS29', 'RPS27', 'RPS12',
        'EEF1A1', 'ACTB', 'GAPDH', 'PSMB2', 'HPRT1', 'UBC'
    ]
}

# Use raw if available
use_raw = adata.raw is not None
gene_universe = set(adata.raw.var_names if use_raw else adata.var_names)

scored_keys = []
for score_name, genes in GENE_SIGNATURES.items():
    present = [g for g in genes if g in gene_universe]
    if len(present) < MIN_GENES_FOR_SCORING:
        continue
    sc.tl.score_genes(
        adata,
        gene_list=present,
        ctrl_size=min(CTRL_SIZE, len(present)),
        score_name=score_name,
        use_raw=use_raw,
    )
    scored_keys.append(score_name)

print(f"Scored: {scored_keys}")


## 7. Score visualization
UMAP and violin summaries.

In [ ]:
if 'X_umap' not in adata.obsm:
    raise ValueError('UMAP embedding missing in adata. Run the core pipeline first.')

# UMAP plots
sc.pl.umap(
    adata,
    color=scored_keys + [GROUP_KEY],
    ncols=3,
    show=False,
)
my_savefig(DIR_FIG_SAVE / 'UMAP_scores.pdf')
plt.show()

# Violin plots by group
sc.pl.violin(
    adata,
    keys=scored_keys,
    groupby=GROUP_KEY,
    jitter=0.4,
    stripplot=True,
    rotation=90,
    show=False,
)
my_savefig(DIR_FIG_SAVE / f'scores_by_{GROUP_KEY}.pdf')
plt.show()


## 8. LOY/ROY classification
Label LOY/ROY and summarize proportions.

In [ ]:
add_loy_roy_labels(
    adata,
    threshold=LOY_THRESHOLD,
    label_col='Y_group',
    low_label='LOY',
    high_label='ROY',
)

counts = build_loy_roy_table(
    adata,
    group_key=GROUP_KEY,
    label_col='Y_group',
    low_label='LOY',
    high_label='ROY',
)
props = add_loy_roy_proportions(counts, low_label='LOY', high_label='ROY')

props.to_csv(DIR_SAVE / 'loy_roy_proportions_by_group.csv')
print(props.head())


## 9. LOY/ROY statistics
Chi-square across groups and Fisher tests vs reference.

In [ ]:
chi2, p_value, dof, expected = chi_square_loy_roy(counts)
print(f'Chi-square p-value: {p_value:.3e}')

pairwise = fisher_pairwise_vs_reference(counts, reference=REFERENCE_GROUP, low_label='LOY', high_label='ROY')
pairwise.to_csv(DIR_SAVE / f'fisher_vs_{REFERENCE_GROUP}.csv')
print(pairwise.sort_values('p_value').head())


## 10. Correlation: Y_score vs scTreeMetRate
Correlation across lineage groups and sample groups.

In [ ]:
plot_score_correlation(
    adata,
    x_key='Y_score',
    y_key='scTreeMetRate',
    group_key='LineageGroup',
    title='Y_score vs scTreeMetRate by Lineage',
    corr_method=CORR_METHOD,
    xlim=X_LIMITS,
    ylim=Y_LIMITS,
    save_path=DIR_FIG_SAVE / 'Y_vs_scTreeMetRate_by_Lineage.pdf',
)

plot_score_correlation(
    adata,
    x_key='Y_score',
    y_key='scTreeMetRate',
    group_key=GROUP_KEY,
    title='Y_score vs scTreeMetRate by Sample',
    corr_method=CORR_METHOD,
    xlim=X_LIMITS,
    ylim=Y_LIMITS,
    save_path=DIR_FIG_SAVE / 'Y_vs_scTreeMetRate_by_Sample.pdf',
)

plot_score_correlation(
    adata,
    x_key='Y_score',
    y_key='scTreeMetRate',
    group_key=GROUP_KEY,
    title='Y_score vs scTreeMetRate by Sample (Excluding LM0)',
    corr_method=CORR_METHOD,
    xlim=X_LIMITS,
    ylim=Y_LIMITS,
    groups_to_exclude=EXCLUDE_GROUPS,
    save_path=DIR_FIG_SAVE / 'Y_vs_scTreeMetRate_by_Sample_Excluding_LM0.pdf',
)


## Figure S6 (Related to Figure 6)
Generate panels A–H as described in the manuscript legend. Outputs are saved under `figures/figure6/03_FigS6_Supplementary/` with `FigS6[A-H]_*.pdf` and `FigS6[A-H]_*.png` filenames.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# -------------------------------
# Parameters (tweakable)
# -------------------------------
S6_EXCLUDE_LINEAGEGROUPS = [18]  # legacy exclusion used in older plotting
LOY_THRESHOLD = float(LOY_THRESHOLD)  # reuse notebook parameter
LOY_ENRICH_PCT = 0.75  # clone-level enrichment threshold (per-site consistency)
S6H_MIN_CELL = 20  # minimum cells per site within a clone to evaluate enrichment
S6H_TREE_SUMMARY = 'median'  # canonical clone-level summary reused by standalone H and the composite panel
META_WEAK_CUTOFF = 0.001
META_HIGH_CUTOFF = 0.008

LOY_COLOR = 'tab:blue'
ROY_COLOR = 'tab:orange'
LOYROY_PALETTE = {'LOY': LOY_COLOR, 'ROY': ROY_COLOR}

# Backward-compat alias: the source dataset uses scTreeMetRate; the manuscript text uses scMetRate
if 'scMetRate' not in adata.obs.columns:
    if 'scTreeMetRate' in adata.obs.columns:
        adata.obs['scMetRate'] = adata.obs['scTreeMetRate']
    else:
        raise KeyError("Neither 'scMetRate' nor 'scTreeMetRate' found in adata.obs")


# Map sample IDs to manuscript labels
ENGRAFT_LABELS = {
    'LM0_NotEngraft': 'Not engrafted',
    'LM0_Engraft': 'Engrafted',
}
SITE_LABELS = {
    'LL': 'LL',
    'Liv': 'Liv',
    'M1': 'M1',
    'M2': 'M2',
    'RE': 'RE',
    'RW': 'RW',
}
# Compartment labels (match the example figure)
# - Panel C legend uses parentheses to indicate merged sites
# - Panels D/F x-axis uses short names
COMPARTMENT_LABELS_DF = {
    'LL': 'Left lung',
    'Liv': 'Liver',
    'M': 'Lymph node',
    'RL': 'Right lung',
    'LM0_Engraft': 'Engrafted',
    'LM0_NotEngraft': 'Not engrafted',
}
COMPARTMENT_LABELS_C = {
    'LM0_NotEngraft': 'Not engrafted',
    'LM0_Engraft': 'Engrafted',
    'LL': 'Left lung',
    'RL': 'Right lung (RE+RW)',
    'M': 'Lymph node (M1+M2)',
    'Liv': 'Liver',
}

# Orders to match the provided example figure
COMPARTMENT_ORDER_D = ['LL', 'Liv', 'M', 'RL', 'LM0_Engraft', 'LM0_NotEngraft']
COMPARTMENT_ORDER_F = ['LM0_NotEngraft', 'LM0_Engraft', 'LL', 'RL', 'M', 'Liv']
COMPARTMENT_ORDER_C = ['LM0_NotEngraft', 'LM0_Engraft', 'LL', 'RL', 'M', 'Liv']

COMPARTMENT_ORDER_D_LABELS = [COMPARTMENT_LABELS_DF[k] for k in COMPARTMENT_ORDER_D]
COMPARTMENT_ORDER_F_LABELS = [COMPARTMENT_LABELS_DF[k] for k in COMPARTMENT_ORDER_F]
COMPARTMENT_ORDER_C_LABELS = [COMPARTMENT_LABELS_C[k] for k in COMPARTMENT_ORDER_C]

# Color scheme to match the example figure
SITE_PALETTE = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']
COMPARTMENT_PALETTE_C = ['tab:brown', 'tab:purple', 'tab:blue', 'tab:red', 'tab:green', 'tab:orange']
COMPARTMENT_PALETTE_DF = {
    'Left lung': 'tab:blue',
    'Liver': 'tab:orange',
    'Lymph node': 'tab:green',
    'Right lung': 'tab:red',
    'Engrafted': 'tab:purple',
    'Not engrafted': 'tab:brown',
}

# Convenience: compartment labels for D and F (different order per panel)
adata.obs['S6_compartment_D'] = adata.obs['Sample2'].astype(str).map(COMPARTMENT_LABELS_DF)
adata.obs['S6_compartment_D'] = pd.Categorical(adata.obs['S6_compartment_D'], categories=COMPARTMENT_ORDER_D_LABELS, ordered=True)
adata.obs['S6_compartment_F'] = adata.obs['Sample2'].astype(str).map(COMPARTMENT_LABELS_DF)
adata.obs['S6_compartment_F'] = pd.Categorical(adata.obs['S6_compartment_F'], categories=COMPARTMENT_ORDER_F_LABELS, ordered=True)

def _umap_view(mask):
    mask = np.asarray(mask)
    sub = ad.AnnData(X=np.zeros((int(mask.sum()), 0), dtype=np.float32))
    sub.obs = adata.obs.loc[mask].copy()
    sub.obsm['X_umap'] = np.asarray(adata.obsm['X_umap'])[mask]
    return sub

sns.set_theme(style='whitegrid', context='paper')

# ---- Plot-ready exports (for Prism/Origin/R, etc.) ----
DIR_PLOT_DATA = DIR_SAVE / 'plot_data'
DIR_PLOT_DATA.mkdir(parents=True, exist_ok=True)
import json
(DIR_PLOT_DATA / 'FigS6_metadata.json').write_text(json.dumps({
    'LOY_THRESHOLD': float(LOY_THRESHOLD),
    'LOY_ENRICH_PCT': float(LOY_ENRICH_PCT),
    'S6H_MIN_CELL': int(S6H_MIN_CELL),
    'S6H_TREE_SUMMARY': S6H_TREE_SUMMARY,
    'META_WEAK_CUTOFF': float(META_WEAK_CUTOFF),
    'META_HIGH_CUTOFF': float(META_HIGH_CUTOFF),
    'SITE_ORDER': site_order if 'site_order' in locals() else ['LL','Liv','M1','M2','RE','RW'],
    'COMPARTMENT_ORDER_D': COMPARTMENT_ORDER_D,
    'COMPARTMENT_ORDER_F': COMPARTMENT_ORDER_F,
    'COMPARTMENT_ORDER_C': COMPARTMENT_ORDER_C,
    'COMPARTMENT_LABELS_DF': COMPARTMENT_LABELS_DF,
    'COMPARTMENT_LABELS_C': COMPARTMENT_LABELS_C,
}, indent=2, sort_keys=True) + '\n')

# -------------------------------
# (A) UMAP of LM0 (pre-implantation) colored by engraftment
# -------------------------------
lm0 = adata.obs['dataset'].astype(str) == 'LM0'
aA = _umap_view(lm0)
aA.obs['Engraftment'] = aA.obs['sampleID'].astype(str).map(ENGRAFT_LABELS)
aA.obs['Engraftment'] = pd.Categorical(aA.obs['Engraftment'], categories=['Not engrafted', 'Engrafted'], ordered=True)
sc.pl.umap(aA, color='Engraftment', palette={'Not engrafted': 'tab:orange', 'Engrafted': 'tab:blue'}, title='pre-implantation', show=False)
my_savefig(DIR_FIG_SAVE / 'FigS6A_UMAP_LM0_engraftment.pdf')
plt.show()

dfA = pd.DataFrame({
    'UMAP1': np.asarray(aA.obsm['X_umap'])[:, 0] if aA.n_obs else [],
    'UMAP2': np.asarray(aA.obsm['X_umap'])[:, 1] if aA.n_obs else [],
    'Engraftment': aA.obs['Engraftment'].astype(str).to_numpy() if aA.n_obs else [],
}, index=aA.obs.index)
dfA.to_csv(DIR_PLOT_DATA / 'FigS6A_LM0_umap_engraftment.csv', index_label='cell')

# -------------------------------
# (B) UMAP of M5k (post-implantation) colored by tissue site
# -------------------------------
m5k = adata.obs['dataset'].astype(str) == 'M5k'
aB = _umap_view(m5k)
aB.obs['Site'] = aB.obs['sampleID'].astype(str).map(SITE_LABELS)
site_order = ['LL','Liv','M1','M2','RE','RW']
aB.obs['Site'] = pd.Categorical(aB.obs['Site'], categories=site_order, ordered=True)
sc.pl.umap(aB, color='Site', palette=SITE_PALETTE, title='post-implantation', show=False)
my_savefig(DIR_FIG_SAVE / 'FigS6B_UMAP_M5k_by_site.pdf')
plt.show()

dfB = pd.DataFrame({
    'UMAP1': np.asarray(aB.obsm['X_umap'])[:, 0] if aB.n_obs else [],
    'UMAP2': np.asarray(aB.obsm['X_umap'])[:, 1] if aB.n_obs else [],
    'Site': aB.obs['Site'].astype(str).to_numpy() if aB.n_obs else [],
}, index=aB.obs.index)
dfB.to_csv(DIR_PLOT_DATA / 'FigS6B_M5k_umap_by_site.csv', index_label='cell')

# -------------------------------
# (C) UMAP of all cells colored by engraftment/metastasis compartment
# -------------------------------
aC = _umap_view(np.ones(adata.n_obs, dtype=bool))
aC.obs['Compartment'] = aC.obs['Sample2'].astype(str).map(COMPARTMENT_LABELS_C)
aC.obs['Compartment'] = pd.Categorical(aC.obs['Compartment'], categories=COMPARTMENT_ORDER_C_LABELS, ordered=True)
sc.pl.umap(aC, color='Compartment', palette=COMPARTMENT_PALETTE_C, title='all cells', show=False)
my_savefig(DIR_FIG_SAVE / 'FigS6C_UMAP_all_by_compartment.pdf')
plt.show()

dfC = pd.DataFrame({
    'UMAP1': np.asarray(aC.obsm['X_umap'])[:, 0] if aC.n_obs else [],
    'UMAP2': np.asarray(aC.obsm['X_umap'])[:, 1] if aC.n_obs else [],
    'Compartment': aC.obs['Compartment'].astype(str).to_numpy() if aC.n_obs else [],
}, index=aC.obs.index)
dfC.to_csv(DIR_PLOT_DATA / 'FigS6C_allcells_umap_by_compartment.csv', index_label='cell')

# -------------------------------
# (D) Violin: X-score and housekeeping score by compartment
# -------------------------------
need = ['X_score', 'Housekeep_score']
missing = [k for k in need if k not in adata.obs.columns]
if missing:
    raise KeyError(f'Missing score(s) in adata.obs: {missing}. Run the gene signature scoring section first.')

dfD = adata.obs[['S6_compartment_D', 'X_score', 'Housekeep_score']].copy()
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2), dpi=300)
sns.violinplot(data=dfD, x='S6_compartment_D', y='X_score', order=COMPARTMENT_ORDER_D_LABELS, hue='S6_compartment_D', palette=COMPARTMENT_PALETTE_DF, dodge=False, cut=0, inner='box', linewidth=1, ax=axes[0])
leg = axes[0].get_legend()
if leg is not None:
    leg.remove()
axes[0].set_title('X-score')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=90)
sns.violinplot(data=dfD, x='S6_compartment_D', y='Housekeep_score', order=COMPARTMENT_ORDER_D_LABELS, hue='S6_compartment_D', palette=COMPARTMENT_PALETTE_DF, dodge=False, cut=0, inner='box', linewidth=1, ax=axes[1])
leg = axes[1].get_legend()
if leg is not None:
    leg.remove()
axes[1].set_title('Housekeeping-score')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=90)
fig.tight_layout(w_pad=2.0)
my_savefig(DIR_FIG_SAVE / 'FigS6D_violin_X_and_housekeeping_by_compartment.pdf')
plt.show()

dfD_export = dfD.copy()
dfD_export['S6_compartment_D'] = dfD_export['S6_compartment_D'].astype(str)
dfD_export.to_csv(DIR_PLOT_DATA / 'FigS6D_scores_by_compartment.csv', index_label='cell')
plt.close(fig)

# -------------------------------
# (E) Histogram of Y-score with threshold
# -------------------------------
fig, ax = plt.subplots(figsize=(4.0, 3.0), dpi=300)
sns.histplot(adata.obs['Y_score'], bins=60, stat='density', color='#cccccc', edgecolor='#666666', ax=ax)
ax.axvline(LOY_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold ({LOY_THRESHOLD:.1f})')
ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.legend(frameon=False, loc='upper right')
my_savefig(DIR_FIG_SAVE / 'FigS6E_hist_Yscore_threshold.pdf')
plt.show()

dfE = adata.obs[['Y_score']].copy()
dfE.to_csv(DIR_PLOT_DATA / 'FigS6E_Yscore_values.csv', index_label='cell')
plt.close(fig)

# -------------------------------
# (F) Cell counts: LOY vs ROY per compartment
# -------------------------------
dfF = adata.obs[['S6_compartment_F', 'Y_group']].copy()
countsF = dfF.groupby(['S6_compartment_F', 'Y_group'], observed=False).size().unstack(fill_value=0)
for col in ['LOY','ROY']:
    if col not in countsF.columns:
        countsF[col] = 0
countsF = countsF[['LOY','ROY']].loc[COMPARTMENT_ORDER_F_LABELS]

fig, ax = plt.subplots(figsize=(5.2, 3.0), dpi=300)
x = np.arange(len(countsF.index))
ax.bar(x - 0.18, countsF['LOY'].values, width=0.35, color=LOY_COLOR, label='LOY')
ax.bar(x + 0.18, countsF['ROY'].values, width=0.35, color=ROY_COLOR, label='ROY')
ax.set_xticks(x)
ax.set_xticklabels(countsF.index, rotation=90)
ax.set_ylabel('Total cell number')
ax.legend(frameon=False, loc='upper right')
my_savefig(DIR_FIG_SAVE / 'FigS6F_counts_LOY_vs_ROY_by_compartment.pdf')
plt.show()

countsF.to_csv(DIR_PLOT_DATA / 'FigS6F_counts_LOY_ROY_by_compartment.csv')
countsF_long = countsF.reset_index().rename(columns={countsF.index.name or 'S6_compartment_F': 'Compartment'})
countsF_long = countsF_long.melt(id_vars=['Compartment'], var_name='LOY_status', value_name='count')
countsF_long.to_csv(DIR_PLOT_DATA / 'FigS6F_counts_LOY_ROY_by_compartment_long.csv', index=False)
plt.close(fig)

# -------------------------------
# (G) Spearman correlation: Y-score vs scMetRate (M5k), shown in gray (no site coloring)
# -------------------------------
dfG = adata.obs.loc[m5k, ['Y_score', 'scMetRate', 'sampleID']].copy()
dfG['site'] = dfG['sampleID'].astype(str)
r, p = spearmanr(dfG['Y_score'], dfG['scMetRate'])

fig, ax = plt.subplots(figsize=(4.2, 3.2), dpi=300)
sns.scatterplot(data=dfG, x='Y_score', y='scMetRate', color='#7f7f7f', s=6, alpha=0.35, linewidth=0, ax=ax, rasterized=True)
# Rasterize dense point cloud to keep PDFs small (axes/line remain vector)
for _c in ax.collections:
    _c.set_rasterized(True)
sns.regplot(data=dfG, x='Y_score', y='scMetRate', scatter=False, color='black', line_kws={'linestyle': '--', 'lw': 1.2, 'alpha': 0.8}, ax=ax)
ax.set_xlabel('Y-score')
ax.set_ylabel('metastatic capacity (scMetRate)')
ax.text(0.98, 0.98, f"Spearman r: {r:.2f}\nP: {p:.2e}", transform=ax.transAxes, ha='right', va='top', fontsize=9, bbox={'boxstyle':'round,pad=0.3', 'fc':'white', 'alpha':0.8})
fig.tight_layout()
my_savefig(DIR_FIG_SAVE / 'FigS6G_spearman_Yscore_vs_scMetRate_by_site.pdf')
plt.show()

dfG_export = dfG[['Y_score', 'scMetRate', 'sampleID']].copy()
dfG_export.to_csv(DIR_PLOT_DATA / 'FigS6G_Yscore_scMetRate_M5k.csv', index_label='cell')
(DIR_PLOT_DATA / 'FigS6G_spearman.json').write_text(json.dumps({'r': float(r), 'p': float(p)}, indent=2) + '\n')
plt.close(fig)

# -------------------------------
# -------------------------------
# (H) Clone-level stacked bar: metastatic phenotype (%) for LOY- vs ROY-enriched clonal populations
# Enrichment rule (min_cell/pct): for each clone, for every site where the clone has >= S6H_MIN_CELL cells,
# the LOY (or ROY) fraction must be >= LOY_ENRICH_PCT; otherwise the clone is classified as Mixed.
# -------------------------------
PHENO_LEGEND_ORDER = ['Non metastatic', 'Weakly metastatic', 'Highly metastatic']
# Stack order (bottom -> top) to match the reference figure: darkred at the bottom, gray on top
PHENO_STACK_ORDER = ['Highly metastatic', 'Weakly metastatic', 'Non metastatic']
PHENO_COLORS = {'Non metastatic': '#e5e5e5', 'Weakly metastatic': 'salmon', 'Highly metastatic': 'darkred'}

site_counts_eligible, clone_enriched = build_s6h_clone_enrichment_table(
    adata.obs,
    exclude_lineage_groups=S6_EXCLUDE_LINEAGEGROUPS,
    min_cells_per_site=S6H_MIN_CELL,
    enrich_pct=LOY_ENRICH_PCT,
    weak_cutoff=META_WEAK_CUTOFF,
    high_cutoff=META_HIGH_CUTOFF,
    tree_summary=S6H_TREE_SUMMARY,
)
summary, pct, fisher_tests = summarize_s6h_clone_phenotypes(
    clone_enriched,
    phenotype_order=PHENO_LEGEND_ORDER,
)
pct.to_csv(DIR_SAVE / 'FigS6H_metapheno_pct_by_enriched_group.tsv', sep='	', index=True, index_label='enriched')
fisher_tests.to_csv(DIR_SAVE / 'FigS6H_fisher_tests.csv', index=False)

fig, ax = plt.subplots(figsize=(2.6, 3.2), dpi=300)
bottom = np.zeros(len(pct.index))
x = np.arange(len(pct.index))
for p in PHENO_STACK_ORDER:
    ax.bar(x, pct[p].fillna(0).values, bottom=bottom, color=PHENO_COLORS[p], label=p)
    bottom += pct[p].fillna(0).values
ax.set_xticks(x)
ax.set_xticklabels(pct.index)
ax.set_ylabel('Percentage (%)')
ax.legend(title='Metastatic\nphenotype', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, fontsize=8)
fig.tight_layout()
my_savefig(DIR_FIG_SAVE / 'FigS6H_stackedbar_metapheno_pct_LOY_vs_ROY_enriched_clones.pdf')
plt.show()
plt.close(fig)

# Plot-ready exports for Fig S6H
site_counts_eligible.reset_index().to_csv(DIR_PLOT_DATA / 'FigS6H_site_counts_eligible.csv', index=False)
clone_enriched.reset_index().to_csv(DIR_PLOT_DATA / 'FigS6H_clone_enrichment_summary.csv', index=False)
summary.to_csv(DIR_PLOT_DATA / 'FigS6H_contingency_clone_counts.csv')
pct.to_csv(DIR_PLOT_DATA / 'FigS6H_metapheno_pct_by_enriched_group.csv')
pct_long = pct.reset_index().rename(columns={'index': 'enriched'})
pct_long = pct_long.melt(id_vars=['enriched'], var_name='metastatic_phenotype', value_name='percent')
pct_long.to_csv(DIR_PLOT_DATA / 'FigS6H_metapheno_pct_by_enriched_group_long.csv', index=False)

## Figure S6 composite (A-H)
This cell assembles panels A-H into a single PDF/PNG pair for convenience. Panel H reuses the same canonical clone-level S6H helper and parameters as the standalone export above. You can still use the individual `FigS6[A-H]_*.pdf` and `FigS6[A-H]_*.png` files for final figure layout.

In [ ]:
# Composite layout for Figure S6 (A-H)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from scipy.stats import spearmanr

# --- Layout mapping (edit here if you want to swap panel positions)
# Uses a 3x6 GridSpec to match the manuscript-style layout:
# Row1: A | B | C  (each spans 2 columns)
# Row2: D (2 small plots) | E | F
# Row3: G (wide) | H
PANEL_SLOTS = {
    'A': (0, slice(0, 2)),
    'B': (0, slice(2, 4)),
    'C': (0, slice(4, 6)),
    'D': (1, slice(0, 2)),
    'E': (1, slice(2, 4)),
    'F': (1, slice(4, 6)),
    'G': (2, slice(0, 4)),
    'H': (2, slice(4, 6)),
}

sns.set_theme(style='whitegrid', context='paper')

need_cols = {'dataset','sampleID','Sample2','Y_group','Y_score','X_score','Housekeep_score','scTreeMetRate','LineageGroup','TreeMetRate'}
missing = [c for c in sorted(need_cols) if c not in adata.obs.columns]
if missing:
    raise KeyError(f'Missing columns in adata.obs: {missing}. Run the earlier sections first.')

if 'scMetRate' not in adata.obs.columns:
    adata.obs['scMetRate'] = adata.obs['scTreeMetRate']

ENGRAFT_LABELS = {'LM0_NotEngraft': 'Not engrafted', 'LM0_Engraft': 'Engrafted'}
SITE_LABELS = {'LL': 'LL','Liv': 'Liv','M1': 'M1','M2': 'M2','RE': 'RE','RW': 'RW'}
SITE_PALETTE = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']
COMPARTMENT_LABELS_DF = {
    'LL': 'Left lung',
    'Liv': 'Liver',
    'M': 'Lymph node',
    'RL': 'Right lung',
    'LM0_Engraft': 'Engrafted',
    'LM0_NotEngraft': 'Not engrafted',
}
COMPARTMENT_LABELS_C = {
    'LM0_NotEngraft': 'Not engrafted',
    'LM0_Engraft': 'Engrafted',
    'LL': 'Left lung',
    'RL': 'Right lung (RE+RW)',
    'M': 'Lymph node (M1+M2)',
    'Liv': 'Liver',
}
COMPARTMENT_PALETTE_C = ['tab:brown', 'tab:purple', 'tab:blue', 'tab:red', 'tab:green', 'tab:orange']
compartment_order_D = ['LL','Liv','M','RL','LM0_Engraft','LM0_NotEngraft']
compartment_order_labels_D = [COMPARTMENT_LABELS_DF[k] for k in compartment_order_D]
compartment_order_F = ['LM0_NotEngraft','LM0_Engraft','LL','RL','M','Liv']
compartment_order_labels_F = [COMPARTMENT_LABELS_DF[k] for k in compartment_order_F]
COMPARTMENT_PALETTE_DF = {
    'Left lung': 'tab:blue',
    'Liver': 'tab:orange',
    'Lymph node': 'tab:green',
    'Right lung': 'tab:red',
    'Engrafted': 'tab:purple',
    'Not engrafted': 'tab:brown',
}
site_order = ['LL','Liv','M1','M2','RE','RW']
compartment_order_C = ['LM0_NotEngraft','LM0_Engraft','LL','RL','M','Liv']
compartment_order_labels_C = [COMPARTMENT_LABELS_C[k] for k in compartment_order_C]

def _umap_view(mask):
    import anndata as ad
    mask = np.asarray(mask)
    sub = ad.AnnData(X=np.zeros((int(mask.sum()), 0), dtype=np.float32))
    sub.obs = adata.obs.loc[mask].copy()
    sub.obsm['X_umap'] = np.asarray(adata.obsm['X_umap'])[mask]
    return sub

def _panel_label(ax, letter: str):
    ax.text(-0.10, 1.05, letter, transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom')

fig = plt.figure(figsize=(12.5, 9.0), dpi=300)
gs = GridSpec(3, 6, figure=fig, height_ratios=[1.05, 0.95, 1.1], hspace=0.55, wspace=0.55)

axA = fig.add_subplot(gs[PANEL_SLOTS['A']])
lm0 = adata.obs['dataset'].astype(str) == 'LM0'
aA = _umap_view(lm0)
aA.obs['Engraftment'] = aA.obs['sampleID'].astype(str).map(ENGRAFT_LABELS)
aA.obs['Engraftment'] = pd.Categorical(aA.obs['Engraftment'], categories=['Not engrafted','Engrafted'], ordered=True)
sc.pl.umap(aA, color='Engraftment', palette={'Not engrafted': 'tab:orange', 'Engrafted': 'tab:blue'}, title='pre-implantation', show=False, ax=axA, legend_loc='right margin')
_panel_label(axA, 'A')

axB = fig.add_subplot(gs[PANEL_SLOTS['B']])
m5k = adata.obs['dataset'].astype(str) == 'M5k'
aB = _umap_view(m5k)
aB.obs['Site'] = aB.obs['sampleID'].astype(str).map(SITE_LABELS)
aB.obs['Site'] = pd.Categorical(aB.obs['Site'], categories=site_order, ordered=True)
sc.pl.umap(aB, color='Site', palette=SITE_PALETTE, title='post-implantation', show=False, ax=axB, legend_loc='right margin')
_panel_label(axB, 'B')

axC = fig.add_subplot(gs[PANEL_SLOTS['C']])
aC = _umap_view(np.ones(adata.n_obs, dtype=bool))
aC.obs['Compartment'] = aC.obs['Sample2'].astype(str).map(COMPARTMENT_LABELS_C)
aC.obs['Compartment'] = pd.Categorical(aC.obs['Compartment'], categories=compartment_order_labels_C, ordered=True)
sc.pl.umap(aC, color='Compartment', palette=COMPARTMENT_PALETTE_C, title='all cells', show=False, ax=axC, legend_loc='right margin')
_panel_label(axC, 'C')

d_slot = gs[PANEL_SLOTS['D']]
subD = d_slot.subgridspec(1, 2, wspace=0.35)
axD1 = fig.add_subplot(subD[0, 0])
axD2 = fig.add_subplot(subD[0, 1])
dfD = adata.obs[['Sample2','X_score','Housekeep_score']].copy()
dfD['Compartment'] = dfD['Sample2'].astype(str).map(COMPARTMENT_LABELS_DF)
dfD['Compartment'] = pd.Categorical(dfD['Compartment'], categories=compartment_order_labels_D, ordered=True)
sns.violinplot(data=dfD, x='Compartment', y='X_score', order=compartment_order_labels_D, hue='Compartment', palette=COMPARTMENT_PALETTE_DF, dodge=False, cut=0, inner='box', linewidth=1, ax=axD1)
leg = axD1.get_legend()
if leg is not None:
    leg.remove()
axD1.set_title('X-score')
axD1.set_xlabel('')
axD1.tick_params(axis='x', rotation=90)
sns.violinplot(data=dfD, x='Compartment', y='Housekeep_score', order=compartment_order_labels_D, hue='Compartment', palette=COMPARTMENT_PALETTE_DF, dodge=False, cut=0, inner='box', linewidth=1, ax=axD2)
leg = axD2.get_legend()
if leg is not None:
    leg.remove()
axD2.set_title('Housekeeping-score')
axD2.set_xlabel('')
axD2.tick_params(axis='x', rotation=90)
_panel_label(axD1, 'D')

axE = fig.add_subplot(gs[PANEL_SLOTS['E']])
sns.histplot(adata.obs['Y_score'], bins=60, stat='density', color='#cccccc', edgecolor='#666666', ax=axE)
axE.axvline(float(LOY_THRESHOLD), color='red', linestyle='--', linewidth=2, label=f'Threshold ({float(LOY_THRESHOLD):.1f})')
axE.set_xlabel('Value')
axE.set_ylabel('Density')
axE.legend(frameon=False, fontsize=8)
_panel_label(axE, 'E')

axF = fig.add_subplot(gs[PANEL_SLOTS['F']])
dfF = adata.obs[['Sample2','Y_group']].copy()
dfF['Compartment'] = dfF['Sample2'].astype(str).map(COMPARTMENT_LABELS_DF)
dfF['Compartment'] = pd.Categorical(dfF['Compartment'], categories=compartment_order_labels_F, ordered=True)
countsF = dfF.groupby(['Compartment','Y_group'], observed=False).size().unstack(fill_value=0)
for col in ['LOY','ROY']:
    if col not in countsF.columns:
        countsF[col] = 0
countsF = countsF[['LOY','ROY']].loc[compartment_order_labels_F]
x = np.arange(len(countsF.index))
axF.bar(x - 0.18, countsF['LOY'].values, width=0.35, color='tab:blue', label='LOY')
axF.bar(x + 0.18, countsF['ROY'].values, width=0.35, color='tab:orange', label='ROY')
axF.set_xticks(x)
axF.set_xticklabels(countsF.index, rotation=90)
axF.set_ylabel('Total cell number')
axF.legend(frameon=False, fontsize=8)
_panel_label(axF, 'F')

axG = fig.add_subplot(gs[PANEL_SLOTS['G']])
dfG = adata.obs.loc[m5k, ['Y_score','scMetRate','sampleID']].copy()
dfG['site'] = dfG['sampleID'].astype(str)
r, p = spearmanr(dfG['Y_score'], dfG['scMetRate'])
sns.scatterplot(data=dfG, x='Y_score', y='scMetRate', color='#7f7f7f', s=5, alpha=0.30, linewidth=0, ax=axG, rasterized=True)
# Rasterize dense point cloud to keep PDFs small (axes/line remain vector)
for _c in ax.collections:
    _c.set_rasterized(True)
sns.regplot(data=dfG, x='Y_score', y='scMetRate', scatter=False, color='black', line_kws={'linestyle': '--', 'lw': 1.2, 'alpha': 0.8}, ax=axG)
axG.set_xlabel('Y-score')
axG.set_ylabel('metastatic capacity (scMetRate)')
axG.text(0.98, 0.98, f"Spearman r: {r:.2f}\nP: {p:.2e}", transform=axG.transAxes, ha='right', va='top', fontsize=9, bbox={'boxstyle':'round,pad=0.3', 'fc':'white', 'alpha':0.8})
_panel_label(axG, 'G')

axH = fig.add_subplot(gs[PANEL_SLOTS['H']])
if 'pct' not in locals():
    _, clone_enriched_h = build_s6h_clone_enrichment_table(
        adata.obs,
        exclude_lineage_groups=S6_EXCLUDE_LINEAGEGROUPS,
        min_cells_per_site=S6H_MIN_CELL,
        enrich_pct=LOY_ENRICH_PCT,
        weak_cutoff=META_WEAK_CUTOFF,
        high_cutoff=META_HIGH_CUTOFF,
        tree_summary=S6H_TREE_SUMMARY,
    )
    _, pct_h, _ = summarize_s6h_clone_phenotypes(
        clone_enriched_h,
        phenotype_order=PHENO_LEGEND_ORDER,
    )
else:
    pct_h = pct.copy()
bottom = np.zeros(len(pct_h.index))
x = np.arange(len(pct_h.index))
for p in PHENO_STACK_ORDER:
    axH.bar(x, pct_h[p].fillna(0).values, bottom=bottom, color=PHENO_COLORS[p], label=p)
    bottom += pct_h[p].fillna(0).values
axH.set_xticks(x)
axH.set_xticklabels(pct_h.index)
axH.set_ylabel('Percentage (%)')
axH.legend(title='Metastatic\nphenotype', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=8)
_panel_label(axH, 'H')

out = DIR_FIG_SAVE / 'FigS6_composite_A_to_H.pdf'
my_savefig(out)
plt.show()
plt.close(fig)